In [1]:
quiet_library <- function(...) { suppressPackageStartupMessages(library(...)) }
quiet_library(hise)
quiet_library(dplyr)
quiet_library(purrr)
quiet_library(nanoparquet)

In [2]:
if(!dir.exists("output")) {
    dir.create("output")
}

### Retrieve sample information

In [3]:
sample_meta_file_uuid = '2da66a1a-17cc-498b-9129-6858cf639caf'

In [4]:
sample_meta_file <- cacheFiles(list(sample_meta_file_uuid))

[1] "downloading fileID 2da66a1a-17cc-498b-9129-6858cf639caf"


In [5]:
sample_meta <- read.csv(sample_meta_file)

In [6]:
nrow(sample_meta)

[1] 108

### Retrieve Cyanno Label information

These files provide ontological relationships between Cyanno cell labels and scRNA-seq labels. We'll incorporate these into the Cyanno label counts files to enable comparison between scRNA-seq and Cyanno results.

In [7]:
label_uuids <- list(
    "PB1" = list("833fe002-4208-4291-9385-15d3dbf23885"),
    "PM1" = list("c621dd24-f681-402d-acc0-35a2b1bf4107"),
    "PS1" = list("201b9eaa-d7a7-477f-99c8-aef19e8bd947"),
    "PT1" = list("af34085f-ea51-4273-aebe-ab38196381fb")
)

In [8]:
label_files <- map(
    label_uuids,
    cacheFiles
)

[1] "downloading fileID 833fe002-4208-4291-9385-15d3dbf23885"
[1] "downloading fileID c621dd24-f681-402d-acc0-35a2b1bf4107"
[1] "downloading fileID 201b9eaa-d7a7-477f-99c8-aef19e8bd947"
[1] "downloading fileID af34085f-ea51-4273-aebe-ab38196381fb"


In [9]:
label_dfs <- map(label_files, read.csv)

## Find flow cytometry data

### Cyanno Labeling Counts
fileType = FlowCytometry-summary-frequency-stats

There should be 4 files for each sample, one per Flow Cytometry panel; 432 in total.

In [10]:
freq_desc <- getFileDescriptors(
    fileType = "FlowCytometry-summary-frequency-stats",
    filter = list(
        sample.sampleKitGuid = as.list(sample_meta[["sample.sampleKitGuid"]])
    )
)

In [11]:
freq_desc <- fileDescToDataframe(freq_desc)
nrow(freq_desc)

[1] 516

Some freq will be from exactly the same aliquots as the scRNA-seq data.

We can match these based on specimen IDs:

In [12]:
freq_desc <- freq_desc %>%
  mutate(pbmc_sample_id = sub(".+(PB[0-9]+-[0-9]+).+", "\\1", file.name)) %>%
  mutate(file_date = sub(".+(20[0-9]{2}-[0-9]{2}-[0-9]{2}).+", "\\1", file.name))

In [13]:
matched_freq_desc <- freq_desc %>%
  filter(pbmc_sample_id %in% sample_meta$pbmc_sample_id)
nrow(matched_freq_desc)

[1] 252

Remove duplicates - in some cases, files were uploaded multiple times. We'll make sure that we take the most recent data available for each sample.

In [14]:
matched_freq_desc <- matched_freq_desc %>%
  arrange(desc(file_date)) %>%
  group_by(pbmc_sample_id, file.panel) %>%
  slice(1)

In [15]:
nrow(matched_freq_desc)

[1] 252

In [16]:
table(matched_freq_desc$file.panel)


PB1 PM1 PS1 PT1 
 64  63  62  63 

In [17]:
length(unique(matched_freq_desc$sample.sampleKitGuid))

[1] 64

In some cases, the specific aliquot may differ, but data may be available from the same batch:

In [18]:
batch_freq_desc <- freq_desc %>%
  filter(!sample.sampleKitGuid %in% matched_freq_desc$sample.sampleKitGuid) %>%
  filter(file.batchID %in% sample_meta$file.batchID) %>%
  arrange(desc(file_date)) %>%
  group_by(sample.sampleKitGuid, file.panel) %>%
  slice(1)
nrow(batch_freq_desc)

[1] 72

In [19]:
table(batch_freq_desc$file.panel)


PB1 PM1 PS1 PT1 
 18  18  18  18 

Some samples were run in separate batches. In these cases, we'll take the most recent file per sample kit for each file panel.

In [20]:
unmatched_freq_desc <- freq_desc %>%
  filter(!sample.sampleKitGuid %in% matched_freq_desc$sample.sampleKitGuid) %>%
  filter(!sample.sampleKitGuid %in% batch_freq_desc$sample.sampleKitGuid) %>%
  arrange(desc(file_date)) %>%
  group_by(sample.sampleKitGuid, file.panel) %>%
  slice(1)
nrow(unmatched_freq_desc)

[1] 104

In [21]:
table(unmatched_freq_desc$file.panel)


PB1 PM1 PS1 PT1 
 26  26  26  26 

In [22]:
selected_freq_desc <- do.call(rbind, list(matched_freq_desc, batch_freq_desc, unmatched_freq_desc))

In [23]:
selected_freq_csv <- paste0("output/human_immune_health_atlas_flow-freq_file-meta_", Sys.Date(), ".csv")
write.csv(
    selected_freq_desc,
    selected_freq_csv,
    row.names = FALSE,
    quote = FALSE
)

In [24]:
table(selected_freq_desc$file.batchID, selected_freq_desc$file.panel)

      
       PB1 PM1 PS1 PT1
  B007   1   1   0   0
  B010   1   1   1   1
  B014   3   3   3   3
  B015   2   2   2   2
  B022   3   3   3   3
  B036   2   2   2   2
  B039   4   4   4   4
  B040   9   9   9   9
  B041   6   6   6   6
  B043   2   2   2   2
  B045   3   3   2   3
  B046   6   5   6   6
  B053   5   5   5   5
  B054   2   2   2   2
  B055   1   1   1   1
  B056   4   4   4   4
  B057   2   2   2   2
  B060   3   3   3   3
  B063   2   2   2   2
  B064   1   1   1   1
  B067   1   1   1   1
  B072   5   5   5   5
  B074   1   1   1   1
  B077   3   3   3   3
  B079   3   3   3   3
  B080   2   2   2   2
  B082   1   1   1   1
  B084   2   2   2   2
  B085   1   1   1   1
  B091   4   4   4   4
  B094   4   4   4   4
  B096   4   4   4   4
  B132   1   1   1   1
  B138   3   3   3   3
  B142   1   1   1   1
  B145   2   2   2   2
  B151   8   8   8   8

Let's split these by panel and retrieve the data.

In [25]:
panel_freq <- split(selected_freq_desc, selected_freq_desc$file.panel)

In [26]:
freq_files <- map(
    panel_freq,
    function(df) {
        cacheFiles(as.list(df[["file.id"]]))
    }
)

ERROR: [1m[33mError[39m in `map()`:[22m
[1m[22m[36mℹ[39m In index: 1.
[36mℹ[39m With name: PB1.
[1mCaused by error in `curl::curl_fetch_memory()`:[22m
[33m![39m Operation was aborted by an application callback


In [ ]:
freq_files[[1]][1:3]

In [ ]:
in_file <- freq_files[[1]][1]
sub("/home/workspace/input/[0-9]+/[^/]+/(.+)/auto.+", "\\1", in_file)

## Build DataFrame for each panel

In [ ]:
panel_dfs <- map(
    freq_files,
    function(file_set) {
        map(file_set,
            function(file_name) {
                df <- read.csv(file_name)
                df$sample.sampleKitGuid <- sub("PB([0-9]+)-.+", "KT\\1", df$sample_id)
                df
            }
        ) %>%
        list_rbind()
    }
)

### Add sample metadata

In [ ]:
panel_dfs <- map(
    panel_dfs,
    left_join,
    sample_meta,
    by = "sample.sampleKitGuid"
)

### Add label conversions

In [ ]:
names(panel_dfs)

In [ ]:
names(panel_dfs[[1]])

In [ ]:
panel_dfs <- map2(
    panel_dfs, label_dfs[names(panel_dfs)],
    left_join,
    by = c("l1_labels", "labels")
)

In [ ]:
head(panel_dfs[[1]])

## Upload data to HISE

In [36]:
study_space_uuid <- "64097865-486d-43b3-8f94-74994e0a72e0"
title <- paste("Imm. Health Atlas Flow .fcs files", Sys.Date())

In [37]:
search_id <- ids::proquint(n_words = 3)

In [43]:
search_id

[1] "dudan-dimuz-porih"

In [38]:
in_list <- as.list(
    selected_freq_desc[["file.id"]]
)

In [39]:
out_list <- c(panel_tars, list(selected_freq_csv))

In [40]:
out_list

$PB1
[1] "output/human_immune_health_atlas_flow-fcs_panel-PB1.tar"

$PM1
[1] "output/human_immune_health_atlas_flow-fcs_panel-PM1.tar"

$PS1
[1] "output/human_immune_health_atlas_flow-fcs_panel-PS1.tar"

$PT1
[1] "output/human_immune_health_atlas_flow-fcs_panel-PT1.tar"

[[5]]
[1] "output/human_immune_health_atlas_flow-fcs_file-meta_2024-10-29.csv"

In [41]:
uploadFiles(
    files = out_list,
    studySpaceId = study_space_uuid,
    title = title,
    inputFileIds = in_list,
    store = "project",
    destination = search_id
)

$Message
[1] "General Okay-ness"

$VisualizationId
[1] "00000000-0000-0000-0000-000000000000"

$AbstractionId
[1] "00000000-0000-0000-0000-000000000000"

$TraceId
[1] "75ef4fa8-fa34-4eef-af91-9e19b3994916"

$ProcessId
[1] "5575e9e0-6115-4e4a-9f52-5fb023e0bce4"

$WorkflowId
[1] "f526ff55-fa34-48fa-8288-3b6562d8cf77"

$FileIds
$FileIds[[1]]
[1] "7fae0c52-fe3c-494c-a4ad-4235a1eb3b41"

$FileIds[[2]]
[1] "87b8dd66-14af-4243-961e-e1eace33202a"

$FileIds[[3]]
[1] "c5fe34ae-4251-4dc9-912d-2dbb85afe418"

$FileIds[[4]]
[1] "81929c4f-d3f4-415d-9761-8e6f40ed617f"

$FileIds[[5]]
[1] "2cc9a15f-28f3-43ab-b2e2-5e7c0d0490a2"

In [42]:
sessionInfo()

R version 4.1.3 (2022-03-10)
Platform: x86_64-conda-linux-gnu (64-bit)
Running under: Ubuntu 24.04 LTS

Matrix products: default
BLAS/LAPACK: /home/workspace/environment/r_flow/lib/libopenblasp-r0.3.20.so

locale:
 [1] LC_CTYPE=C.UTF-8    LC_NUMERIC=C        LC_TIME=C          
 [4] LC_COLLATE=C        LC_MONETARY=C       LC_MESSAGES=C      
 [7] LC_PAPER=C          LC_NAME=C           LC_ADDRESS=C       
[10] LC_TELEPHONE=C      LC_MEASUREMENT=C    LC_IDENTIFICATION=C

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
[1] purrr_0.3.4 dplyr_1.0.8 hise_2.16.0

loaded via a namespace (and not attached):
 [1] Rcpp_1.0.8.3     plyr_1.8.7       pillar_1.7.0     compiler_4.1.3  
 [5] base64enc_0.1-3  bitops_1.0-7     tools_4.1.3      digest_0.6.29   
 [9] uuid_1.1-0       jsonlite_1.8.0   evaluate_0.15    lifecycle_1.0.1 
[13] tibble_3.1.6     pkgconfig_2.0.3  rlang_1.0.2      IRdisplay_1.1   
[17] cli_3.2.0        DBI